# CVLab: Deep Learning Classification und Explainable AI

## Pre-Trained CNN and Grad-CAM

S. Suter, S. Bonato

In this exercise, we model a pre-trained deep learning model for image classification

#### Learning Objectives

By the end of this lab, you should be able to:

- Build and train a binary image classifier using a pre-trained MobileNetV2 in PyTorch. (Apply – L3)

- Freeze/unfreeze layers and justify transfer learning choices. (Analyze – L4)

- Evaluate performance with accuracy, precision, recall, and F1-score. (Evaluate – L5)

- Visualize what the model “sees” using Grad-CAM and interpret its attention maps. (Evaluate – L5)

Reference: this notebook was inspired by https://www.tensorflow.org/tutorials/images/transfer_learning

## 1. Task and Data

In this assignment, we will create and tune a convolutional neural network (CNN) that is able to detect the gender of a given face image. That is, we model a binary classification task. You will be gently guided through every step of the CNN model training and evaluation. We will use a pre-trained image classification model for this task.

The data consists of images containing male and female pictures (head shots) sampled from [Labeled faces in the wild](http://vis-www.cs.umass.edu/lfw/).

*Gary B. Huang, Manu Ramesh, Tamara Berg, and Erik Learned-Miller. Labeled Faces in the Wild: A Database for Studying Face Recognition in Unconstrained Environments. University of Massachusetts, Amherst,
Technical Report 07-49, October, 2007.*

The face images were for teaching purposes prepared in a repository splitting male and female faces such that they are already sorted in two directories for our two classes to predict. The labels for the classes `female` and `male` are hence given by the respective folders containing either male or female pictures. Note, as I classified the images in a semi-automated fashion, it is possible that you depict an occasional misclassified image.

### 1.1. Install Libraries and Import Data

Retrieve the data set used for this exercise. We will import the data directly from a git repository.

First, let's install the required packages.

In [3]:
%pip install gitpython grad-cam opencv-python

  Using cached grad_cam-1.5.5-py3-none-any.whl
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached ttach-0.0.3-py3-none-any.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 3.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 43.8 MB/s eta 0:00:00:00:0100:01
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached ttach-0.0.3-py3-none-any.whl (9.8 kB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    resnet50, ResNet50_Weights,
    inception_v3, Inception_V3_Weights,
)
from sklearn import metrics

from git import Repo
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [13]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
if IN_COLAB:
    DATA_PATH = Path('/content/drive/MyDrive/cvlab')
else:
    DATA_PATH = Path.cwd() / 'data'

DATA_PATH.mkdir(exist_ok=True, parents=True)


FACES_PATH = DATA_PATH / 'faces'
female_path = FACES_PATH / 'female'
male_path = FACES_PATH / 'male'

TRAIN_PATH = FACES_PATH / 'train'
BENCHMARK_PATH = FACES_PATH / 'benchmark'

female_train_path = TRAIN_PATH / 'female'
male_train_path = TRAIN_PATH / 'male'


In [7]:
if FACES_PATH.is_dir():
    shutil.rmtree(FACES_PATH)

Repo.clone_from('https://github.com/susuter/faces_red.git', FACES_PATH)

TRAIN_PATH.mkdir(exist_ok=True, parents=True)

shutil.move(female_path, female_train_path)
shutil.move(male_path, male_train_path)


PosixPath('/mnt/nas05/clusterdata01/group/vised/sds_workshop/simone/cvlab_notebooks/data/faces/train/male')

### 1.2. Load Data Into PyTorch DataLoader
PyTorch offers powerful functions to load and process data. In this notebook we will load the data using `torchvision.datasets.ImageFolder` and `torch.utils.data.DataLoader`. These objects can then be fed directly into the model during training.

Select the parameters below before loading the data. Since the computing capacity on JupyterHub is limited, it is recommended to use a smaller image size. Use square input images.

In [8]:
seed = 42

# ## BEGIN SOLUTION
BATCH_SIZE = 32 # Number of training examples per training step.
IMG_SIZE = (320, 320) # All images are cropped to this size.
VALIDATION_SPLIT = 0.2 # X% of the training data is used for validation.
# ## END SOLUTION

torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# Define transforms for data loading (basic resize and tensor conversion)
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

full_train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=transform)

train_size = int((1 - VALIDATION_SPLIT) * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, validation_dataset = random_split(full_train_dataset, [train_size, val_size],
                                                   generator=torch.Generator().manual_seed(seed))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

benchmark_dataset = datasets.ImageFolder(BENCHMARK_PATH, transform=transform)
benchmark_loader = DataLoader(benchmark_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_names = full_train_dataset.classes
class_values = full_train_dataset.class_to_idx

print(f"Class names: {class_names}")
print(f"Class values: {class_values}")

Class names: ['female', 'male']
Class values: {'female': 0, 'male': 1}


#### 1.2.1. Look at the Data

In [9]:
def scroll_samples(dataset, class_names):
    def show_image(i):
        img, label = dataset[i]  # (C, H, W) tensor, int label

        # Convert CHW -> HWC if it's a tensor
        if hasattr(img, "permute"):
            img = img.permute(1, 2, 0).numpy()
        else:
            img = np.array(img)

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(f"Index: {i}, Label: {class_names[label]}")
        plt.axis("off")
        plt.show()

    slider = widgets.IntSlider(
        min=0,
        max=len(dataset) - 1,
        step=1,
        value=0,
        description="Index",
        continuous_update=False,
    )
    return widgets.interact(show_image, i=slider)


In [ ]:
scroll_samples(train_dataset, class_names)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Index', max=3781), Output()), _…

<function __main__.scroll_samples.<locals>.show_image(i)>

In [ ]:
scroll_samples(validation_dataset, class_names)

In [ ]:
scroll_samples(benchmark_dataset, class_names)

### 1.3. Data preprocessing
As you know from the previous exercise, the images need to be pre-processed in some way before they can be fed into the model. Since we use a pre-trained model (MobileNetV2), we need to normalize the images using ImageNet statistics (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) which the model was trained on.

The preprocessing will be applied in the model's forward pass.

In [14]:
# We'll use the standard ImageNet normalization
preprocess_input = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

### 1.4 Labels
In PyTorch, we use integer labels directly (0 and 1) rather than one-hot encodings. The loss function (CrossEntropyLoss) will handle the conversion internally. This is more memory efficient and is the standard approach in PyTorch.

## 2. Creating and training the model
### 2.1. Creation of the base model
In this task we load a pre-trained model. When using a pre-trained model, we only need to train the classifier. This means we only need to train the last layer of the model. The weights of the other layers are used pre-trained. PyTorch's torchvision provides pre-trained models that we can load and modify.

In [15]:
# Load pre-trained MobileNetV2 without the top classification layer
base_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
# Remove the classifier head (we'll add our own)
base_model.classifier = nn.Identity()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /home2/simone/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 216MB/s]


#### 2.1.1. Freeze the Base Model
Since we are using a pre-trained model, we do not need to retrain the weights from the base model. In PyTorch, we freeze weights by setting `requires_grad = False` for all parameters in the base model.

In [16]:
# ## BEGIN SOLUTION
# Freeze all parameters in the base model
for param in base_model.parameters():
    param.requires_grad = False
# ## END SOLUTION

<font color='green'>Task (Optional)</font>: Try unfreezing some of the last layers of the network (or all, why not?) and see how the perfomance changes.

Does the model get better at classification? Or does it forget all that was learned in the pre-training? (Hint: change the learning rate accordingly too)

In [17]:
# Optional, for later
unfreeze_for_xai = True
if unfreeze_for_xai:
  for param in base_model.features[-1].parameters():
     param.requires_grad = True

### 2.2. Architecture of the model
You can use the `summary()` function to look at the architecture of the model. As you can see, the MobileNetV2 model has 157 hidden layers and has approximately 2.5 million parameters. This is of course very big, but we don't have to retrain them!

In [18]:
# Print model architecture
print(base_model)
# Count parameters
total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

### 2.3. Feature Extraction
Feature extraction in a pre-trained image model is the process by which the model extracts patterns and important features from an image to make it understandable to the computer. The weights learned correspond to the features found. This means that we will use the model loaded above to extract features from the images. In PyTorch, we pass images through the model's feature extraction layers to get these learned representations.

In [19]:
# Get a batch and extract features
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create a temporary model to test feature extraction
temp_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
temp_model.classifier = nn.Identity()
temp_model.eval()
temp_model = temp_model.to(device)

with torch.no_grad():
    image_batch, label_batch = next(iter(train_loader))
    # Apply preprocessing (normalize)
    image_batch_normalized = preprocess_input(image_batch)
    # Move to device if available
    image_batch_normalized = image_batch_normalized.to(device)
    # Extract features (this will go through features -> avgpool -> Identity)
    feature_batch = temp_model(image_batch_normalized)
    # The output might be [batch, 1280, 1, 1], so we flatten it

    # if len(feature_batch.shape) > 2:
    #     feature_batch = torch.flatten(feature_batch, 1)

    print(f"\nImage batch shape: {image_batch.shape}")
    print(f"Feature batch shape: {feature_batch.shape}")
    print(f"Label batch shape: {label_batch.shape}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home2/simone/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 69.2MB/s]



Image batch shape: torch.Size([32, 3, 320, 320])
Feature batch shape: torch.Size([32, 1280])
Label batch shape: torch.Size([32])


As you can see in the above section, an image of size (IMAGE_SIZE, 3) is passed through the MobileNetV2 feature extraction layers. The convolutional layers extract spatial features, and after global average pooling, we get a feature vector of size 1280. This represents the learned features from the pre-trained model that we can use for our classification task.

### 2.4. Add classification layer
Essentially, we add our own layer to the pre-trained model at the end, which classifies the classes we want. For each class, the logits (raw scores before softmax) are output at the end of the CNN. Typically there is one output node per class. During training, CrossEntropyLoss will apply softmax internally, but for inference we can apply softmax to get probabilities.


In [20]:
# ## BEGIN SOLUTION
NUM_CLASSES = 2
# Note: In PyTorch, we don't apply softmax in the model - it's included in CrossEntropyLoss
# But for inference, we can use softmax or log_softmax
# ## END SOLUTION

# MobileNetV2 outputs 1280 features after global average pooling
# So we just need to add a classification layer
prediction_layer = nn.Linear(feature_batch.shape[1], NUM_CLASSES).to(device)

# Test the layers
with torch.no_grad():
    prediction_batch = prediction_layer(feature_batch)
    print(f"Feature batch shape (after base model): {feature_batch.shape}")
    print(f"Prediction batch shape: {prediction_batch.shape}")

Feature batch shape (after base model): torch.Size([32, 1280])
Prediction batch shape: torch.Size([32, 2])


### 2.5. Assembling the model
In this step we now assemble the model from the components defined above. We set regularization parameters such as dropout to 0 in the first phase until we have a reasonably functioning model. You can then set the parameter to a value between 0 and 1.

<font color='green'>Task</font>: try making the classifier more complex, instead of using a single linear layer.
Example: add several several hidden layers, activation functions, or dropout and see if the performance increases.

In [21]:
class FaceClassifier(nn.Module):
    """MobileNetV2-based face classifier. Outputs raw logits of shape (batch, num_classes)."""
    def __init__(self, backbone, num_classes=2, dropout=0.0):
        super().__init__()
        # MobileNetV2: backbone.features → (batch, 1280, H, W); avgpool collapses spatial dims
        self.features = backbone.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = preprocess_input(x)
        x = self.features(x)         # (batch, 1280, H, W)
        x = self.avgpool(x)          # (batch, 1280, 1, 1)
        x = torch.flatten(x, 1)      # (batch, 1280)
        x = self.dropout(x)
        return self.classifier(x)    # (batch, num_classes)


In [22]:
class FaceClassifierResNet(nn.Module):
    """ResNet50-based face classifier. Outputs raw logits of shape (batch, num_classes).

    ResNet50 (with fc=Identity) already applies global avg pooling and flattening internally,
    producing a feature vector of size 2048.
    """
    def __init__(self, backbone, num_classes=2, dropout=0.0):
        super().__init__()
        backbone.fc = nn.Identity()   # remove original classification head
        self.features = backbone      # full ResNet50 → (batch, 2048)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = preprocess_input(x)
        x = self.features(x)          # (batch, 2048)  — avgpool+flatten inside ResNet
        x = self.dropout(x)
        return self.classifier(x)     # (batch, num_classes)


In [23]:
### Choose which pre-trained backbone to use

# Option A: MobileNetV2 (lighter, ~3.4M params)
backbone = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
backbone.classifier = nn.Identity()
model = FaceClassifier(backbone, num_classes=NUM_CLASSES, dropout=0.0)

# Option B: ResNet50 (deeper, ~25M params) — uncomment and comment Option A above
# backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
# model = FaceClassifierResNet(backbone, num_classes=NUM_CLASSES, dropout=0.0)

# Freeze backbone weights — only the classifier head will be trained
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)
print(f"Model: {type(model).__name__}")
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")


Model: FaceClassifier
Total params: 2,226,434 | Trainable: 2,562


In [24]:
# Helper functions for training and evaluation

def evaluate_model(model, dataloader, criterion, device, dataset_name=""):
    """
    Evaluate a model on a given dataloader with a progress bar.
    """
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {dataset_name}", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            running_total += labels.size(0)
            running_correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(dataloader)
    accuracy = running_correct / running_total

    if dataset_name:
        print(f'{dataset_name} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

    return {'loss': avg_loss, 'accuracy': accuracy}


def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    """
    Train a PyTorch model with tqdm progress bars for both training and validation.
    """
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        print(f"\nEpoch [{epoch+1}/{epochs}]")
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            running_total += labels.size(0)
            running_correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = running_correct / running_total

        val_results = evaluate_model(model, val_loader, criterion, device, dataset_name="Validation")
        val_loss = val_results['loss']
        val_acc = val_results['accuracy']

        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
        print('-' * 50)

    return history


def get_predictions(model, dataloader, device, return_images=False):
    """
    Get predictions, labels, and optionally images from a dataloader with tqdm.
    """
    model.eval()
    all_predictions = []
    all_labels = []
    all_probs = []
    all_images = [] if return_images else None

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Getting predictions", leave=False):
            images_device = images.to(device)
            outputs = model(images_device)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

            if return_images:
                all_images.append(images.numpy())

    result = {
        'predictions': np.array(all_predictions),
        'labels': np.array(all_labels),
        'probabilities': np.array(all_probs)
    }

    if return_images:
        result['images'] = np.concatenate(all_images, axis=0)

    return result


def plot_training_history(history):
    """
    Plot training and validation accuracy and loss curves.

    Args:
        history: Dictionary with 'loss', 'accuracy', 'val_loss', 'val_accuracy' keys
    """
    acc = history['accuracy']
    val_acc = history['val_accuracy']
    loss = history['loss']
    val_loss = history['val_loss']

    plt.figure(figsize=(8, 8))
    plt.subplot(2, 1, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.ylabel('Accuracy')
    plt.ylim([min(plt.ylim()), 1])
    plt.title('Training and Validation Accuracy')

    plt.subplot(2, 1, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.ylabel('Cross Entropy')
    plt.ylim([0, 1.0])
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.tight_layout()
    plt.show()


def calculate_metrics(y_true, y_pred):
    """
    Calculate classification metrics (precision, recall, accuracy, F1).

    Args:
        y_true: True labels
        y_pred: Predicted labels

    Returns:
        Dictionary with metrics
    """

    precision = metrics.precision_score(y_true, y_pred, average='macro')
    recall = metrics.recall_score(y_true, y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_true, y_pred)
    f1 = metrics.f1_score(y_true, y_pred, average='macro')

    return {
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy,
        'f1': f1
    }


def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix'):
    """
    Plot confusion matrix as a heatmap.

    Args:
        y_true: True labels
        y_pred: Predicted labels
        class_names: List of class names
        title: Title for the plot
    """

    cf_matrix = metrics.confusion_matrix(y_true, y_pred)

    sns.heatmap(cf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)

    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title(title)
    plt.tight_layout()
    plt.show()


### 2.6. Model training
Now we train the model on our data set. Choose the hyperparameters yourself here, such as the number of epochs, learning rates, etc. At the end of the training, we look at the accuracy and loss curves again.


In [25]:
# ## BEGIN SOLUTION
EPOCHS = 3
LEARNING_RATE = 0.0001
# ## END SOLUTION

<font color='green'>Task (Optional)</font>: try training using a different optimizer, loss (might need more code adaptation) or learning rate and see what happens.

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Move model to device
model = model.to(device)

In [ ]:
# Train the model using the reusable function
history = train_model(model, train_loader, validation_loader, criterion, optimizer, device, EPOCHS)

In [ ]:
# Evaluate on validation set using the reusable function
val_results = evaluate_model(model, validation_loader, criterion, device, "Validation")

loss0 = val_results['loss']
accuracy0 = val_results['accuracy']

In [ ]:
# Plot training history using the reusable function
plot_training_history(history)

In [ ]:
# Save the model
torch.save(model.state_dict(), "model_faces_v1.pth")
print("Model saved to model_faces_v1.pth")

### 2.7. Cross validation
Perform your training three times and compare the results. What values ​​do you get? How stable is the training? Do the results in the graphics meet your expectations?

## 3. Evaluation with Independent Benkmark Test Set


Now, the trained CNN model is evaluated on an independent test data set, which was not used for to calculated the loss during the training.

To differentiate from the terminology test set, which was used during the training, we use here the terminology benchmark test set.

### 3.1. Loading Data

### 3.2. Prediction on Benchmark Test Set
Now, we calculate the predictions for the benchmark test set given the trained model. The results are the calculated probabilites assigned to either of the class. In a second step, we threshold the probabilities and hence assign the prediction to one class only.

In [ ]:
# Evaluate on benchmark set using the reusable function
benchmark_results = evaluate_model(model, benchmark_loader, criterion, device, "Benchmark")
loss = benchmark_results['loss']
accuracy = benchmark_results['accuracy']

### 3.3. Performance of Benchmark Set

Evaluate the performance on the benchmark data set using the confusion matrix and other key figures such as F1 score etc.

In [ ]:
# Retrieve predictions from the benchmark set using the reusable function
benchmark_data = get_predictions(model, benchmark_loader, device, return_images=True)

predictions = benchmark_data['predictions']
labels_benchmark = benchmark_data['labels']
images_benchmark = benchmark_data['images']

print('Predictions (first 10):\n', predictions[:10])
print('Labels (first 10):\n', labels_benchmark[:10])

# Visualize some images
plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    random_index = np.random.randint(0, len(labels_benchmark))
    # Convert from CHW to HWC
    img = images_benchmark[random_index].transpose(1, 2, 0)
    plt.imshow(img)
    plt.title(class_names[labels_benchmark[random_index]])
    plt.axis("off")
plt.suptitle('Some images from the benchmark dataset with their labels')
plt.tight_layout()
plt.show()

In [ ]:
# ## BEGIN SOLUTION
# Predictions are already class indices (0 or 1), no need for thresholding
y_test_pred = predictions
# ## END SOLUTION

# Plot confusion matrix using the reusable function
plot_confusion_matrix(labels_benchmark, y_test_pred, class_names, 'Confusion Matrix')


In [ ]:
# ## BEGIN SOLUTION
# Calculate metrics using the reusable function
metrics_dict = calculate_metrics(labels_benchmark, y_test_pred)
# ## END SOLUTION

print('Precision =', metrics_dict['precision'])
print('Recall =', metrics_dict['recall'])
print('F1 score =', metrics_dict['f1'])
print('Accuracy =', metrics_dict['accuracy'])

How do you interpret the results specifically for the use case of gender prediction based on images? What do the calculated key figures mean? Do the numbers match your expectations? Describe your observations and discuss your results.

### 3.4. Visualization of the results
For a few images of each class, output the probability of belonging to a class calculated by the CNN. Choose examples that show clear results and examples that show uncertain predictions. Interpret your results using the selected examples in 2-3 sentences.

In [ ]:
def get_missclassified(dataloader, model, device):
    misclassified = []
    model.eval()
    with torch.no_grad():
        for X, y_true in dataloader:
            X_device = X.to(device)
            y_pred = model(X_device)
            # Get probabilities
            probs = torch.nn.functional.softmax(y_pred, dim=1)
            pred_labels = torch.argmax(y_pred, dim=1).cpu().numpy()
            true_labels = y_true.numpy()

            misclassified_indices = np.where(true_labels != pred_labels)[0]
            if misclassified_indices.size > 0:
                for index in misclassified_indices:
                    misclassified.append((
                        X[index].numpy(),  # Image
                        y_true[index].item(),  # True label
                        pred_labels[index],  # Predicted label
                        probs[index].cpu().numpy()  # Probabilities
                    ))

    print(f'Number of misclassified images: {len(misclassified)}')
    return misclassified

def get_correctly_classified(dataloader, model, device):
    correctly_classified = []
    model.eval()
    with torch.no_grad():
        for X, y_true in dataloader:
            X_device = X.to(device)
            y_pred = model(X_device)
            # Get probabilities
            probs = torch.nn.functional.softmax(y_pred, dim=1)
            pred_labels = torch.argmax(y_pred, dim=1).cpu().numpy()
            true_labels = y_true.numpy()

            correctly_classified_idx = np.where(true_labels == pred_labels)[0]
            if correctly_classified_idx.size > 0:
                for index in correctly_classified_idx:
                    correctly_classified.append((
                        X[index].numpy(),  # Image
                        y_true[index].item(),  # True label
                        pred_labels[index],  # Predicted label
                        probs[index].cpu().numpy()  # Probabilities
                    ))

    print(f'Number of correctly classified images: {len(correctly_classified)}')
    return correctly_classified


def show_example_prediction(classified):
    index = np.random.choice(len(classified))
    print(f'Image with index {index}:')
    # Convert from CHW to HWC
    img = classified[index][0].transpose(1, 2, 0)
    plt.imshow(img)
    print(f'True label: {class_names[classified[index][1]]} ({classified[index][1]})')
    print(f'Predicted label: {class_names[classified[index][2]]} ({classified[index][2]})')
    print(f'Predicted probabilities: {classified[index][3]}')


### 3.4.1 Visualization of incorrectly classified images
Visualize the misclassified images. Interpret your results using the selected examples in 2-3 sentences.

In [ ]:
miss_classified = get_missclassified(benchmark_loader, model, device)

In [ ]:
# ## BEGIN SOLUTION
# Use the show_misclassified function to output a randomly selected image that was misclassified
show_example_prediction(miss_classified)
# ## END SOLUTION

### 3.4.2 Visualization of correctly classified images
Visualize the correctly classified images. Interpret your results using the selected examples in 2-3 sentences.

In [ ]:
classified = get_correctly_classified(benchmark_loader, model, device)

In [ ]:
# ## BEGIN SOLUTION
# Use the show_misclassified function to output a randomly selected image that was misclassified
show_example_prediction(classified)
# ## END SOLUTION

### 3.4.3 Visualization of results using Explainable AI
This section is exploratory and should not be taken as the sole truth. Grad-CAM is a method to visualize the activations of the last layer of a CNN for a specific class using a HeatMap. However, it is not the only method and does not always display helpful heat maps. Nevertheless, it can provide insight into which parts of the image the predictions were based on. The interpretation of the results must be made by people. Play with it yourself. In the task below you can visualize the Grad-CAM heatmaps for the incorrectly (and also correctly) classified images. Find a few exciting examples and interpret the results. See: Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.

In [ ]:
def get_explanation(image_tensor, model, target_layer, class_index):
    """
    Get Grad-CAM explanation for an image.

    Args:
        image_tensor: Input image tensor (C, H, W) in [0, 1] range
        model: PyTorch model
        target_layer: The layer to generate Grad-CAM for (e.g., model.base_model.features[-1])
        class_index: Target class index. If None, uses the predicted class.
    """
    # Instantiate Grad-CAM
    cam = GradCAM(model=model, target_layers=[target_layer])

    # Convert to batch format and move to device
    input_tensor = image_tensor.unsqueeze(0).to(device)

    # Get explanation
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(class_index)])
    grayscale_cam = grayscale_cam[0, :]  # Remove batch dimension

    return grayscale_cam

def show_example_prediction_xai(classified, model):
    model.eval()
    index = np.random.choice(len(classified))
    print(f'Image with index {index}:')

    # Get the image and convert to tensor
    img_np = classified[index][0]  # Shape: (C, H, W) in [0, 1]
    img_tensor = torch.from_numpy(img_np).float()

    # Get the target layer for Grad-CAM (last convolutional layer in MobileNetV2)
    # MobileNetV2 features: the last conv layer before global pooling
    target_layer = model.features[-1]

    # Get predicted class
    predicted_class = np.argmax(classified[index][3])
    true_label = classified[index][1]

    # Get Grad-CAM explanation
    explanation = get_explanation(img_tensor, model, target_layer, class_index=predicted_class)

    # Convert image to HWC format for visualization (in [0, 1] range)
    img_rgb = img_np.transpose(1, 2, 0)

    # Normalize explanation to [0, 1] for visualization
    explanation_normalized = (explanation - explanation.min()) / (explanation.max() - explanation.min() + 1e-8)

    # Create overlay
    visualization = show_cam_on_image(img_rgb, explanation_normalized, use_rgb=True)

    plt.figure(figsize=(23, 5))

    # Display original image
    plt.subplot(1, 3, 1)
    plt.imshow(img_rgb)
    plt.title(f'True label: {class_names[true_label]} ({true_label}) - Predicted: {class_names[predicted_class]} ({predicted_class})')
    plt.axis('off')

    # Display Grad-CAM explanation overlaid
    plt.subplot(1, 3, 2)
    plt.imshow(visualization)
    plt.title('Grad-CAM Overlaid')
    plt.axis('off')

    # Display only Grad-CAM
    plt.subplot(1, 3, 3)
    plt.imshow(explanation, cmap='viridis')
    plt.title('Only Grad-CAM')
    plt.colorbar(label='Importance')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

## NB: Remember to unfreeze the last layer, else they won't produce any gradients

<font color='green'>Task</font>: are the feature maps interpretable? What can you tell?

In [ ]:
# ## BEGIN SOLUTION
show_example_prediction_xai(miss_classified, model)
# ## END SOLUTION

In [ ]:
# ## BEGIN SOLUTION
show_example_prediction_xai(classified, model)
# ## END SOLUTION